## 01a_ingest_cms_api — RCM Intelligence Platform
### Purpose
Ingests live Medicare inpatient and outpatient claims data from the 
CMS Open Data API into Bronze Delta tables.

### Data sources
- CMS Medicare Inpatient Hospitals by Provider & Service (live API)
- CMS Medicare Outpatient Hospitals by Provider & Service (live API)

### What this does
1. Fetches all pages from CMS inpatient API
2. Adds audit metadata columns to every row
3. Writes raw data to Bronze Delta table (append mode)
4. Runs same process for outpatient data
5. Logs run to audit table

### Outputs
- rcm_platform.rcm_bronze.inpatient_claims_raw
- rcm_platform.rcm_bronze.outpatient_claims_raw

### Notes
- No transformations here — raw data only
- Safe to re-run — duplicate detection via _batch_id
- Schema inference is on — new CMS columns are handled automatically

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Bronze |
| Run order  | 2 — after 00_setup_databases |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# Every ingestion run gets a unique batch ID
# This lets us trace exactly which run wrote which rows
# ============================================================

import uuid
from datetime import datetime, timezone

BATCH_ID         = str(uuid.uuid4())
BATCH_DATE       = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP  = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME    = "01a_ingest_cms_api"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Batch timestamp : {BATCH_TIMESTAMP}")

In [0]:
# ============================================================
# STEP 1 — DISCOVER CURRENT CMS API ENDPOINTS
# Reads the CMS data.json catalog to find the current URL
# for each dataset — no hardcoded UUIDs that go stale
# ============================================================

import requests

def get_cms_dataset_url(search_term: str) -> str:
    """
    Searches the CMS Open Data catalog for a dataset by title
    and returns the current API endpoint URL.

    Args:
        search_term : partial title to search for in the CMS catalog

    Returns:
        str — the API or CSV download URL for that dataset
    """
    print(f"Searching CMS catalog for: '{search_term}'...")

    resp = requests.get(CMS_DATA_CATALOG_URL, timeout=API_TIMEOUT_SECS)
    resp.raise_for_status()
    catalog  = resp.json()
    datasets = catalog.get("dataset", [])

    # Find exact title match first, then partial
    matches = [
        d for d in datasets
        if search_term.lower() == d.get("title", "").lower()
    ]
    if not matches:
        matches = [
            d for d in datasets
            if search_term.lower() in d.get("title", "").lower()
        ]

    if not matches:
        raise ValueError(
            f"No dataset found matching: '{search_term}'\n"
            f"Total datasets in catalog: {len(datasets)}"
        )

    dataset = matches[0]
    print(f"  Found    : {dataset['title']}")

    # Prefer API format, fall back to CSV
    for fmt in ["API", "CSV"]:
        for dist in dataset.get("distribution", []):
            if dist.get("format", "").upper() == fmt:
                url = dist.get("downloadURL") or dist.get("accessURL", "")
                if url:
                    print(f"  Format   : {fmt}")
                    print(f"  URL      : {url[:80]}...")
                    return url

    raise ValueError(f"No downloadable format found for: '{search_term}'")


# Discover both endpoints
print("=" * 55)
print("  DISCOVERING CMS API ENDPOINTS")
print("=" * 55)

inpatient_url  = get_cms_dataset_url(CMS_INPATIENT_SEARCH_TERM)
outpatient_url = get_cms_dataset_url(CMS_OUTPATIENT_SEARCH_TERM)

print(f"\nEndpoints ready:")
print(f"  Inpatient  : {inpatient_url[:70]}...")
print(f"  Outpatient : {outpatient_url[:70]}...")

In [0]:
# ============================================================
# STEP 2 — INGEST CMS INPATIENT DATA
# Fetch all pages, add audit metadata, write to Bronze Delta
# ============================================================

print("=" * 55)
print("  INGESTING CMS INPATIENT DATA")
print("=" * 55)

# Fetch all records from CMS API
inpatient_records = fetch_cms_paginated(
    url     = inpatient_url,
    limit   = API_PAGE_LIMIT,
    timeout = API_TIMEOUT_SECS
)

# Convert to Spark DataFrame
df_inpatient_raw = spark.createDataFrame(inpatient_records)

# Add audit metadata — never modify raw values, only append columns
df_inpatient_bronze = df_inpatient_raw \
    .withColumn("_source",           lit("cms_api_inpatient")) \
    .withColumn("_batch_id",         lit(BATCH_ID)) \
    .withColumn("_batch_date",       lit(BATCH_DATE)) \
    .withColumn("_ingested_at",      lit(BATCH_TIMESTAMP).cast("timestamp")) \
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME)) \
    .withColumn("_pipeline_version", lit(PIPELINE_VERSION))

# Write to Bronze — append mode preserves full history
inpatient_rows = write_delta(
    df         = df_inpatient_bronze,
    table_name = TBL_BRONZE_INPATIENT_RAW,
    mode       = "append",
    comment    = "Raw CMS Medicare inpatient claims by provider and DRG — append only, never modified"
)

print(f"\nBronze table : {TBL_BRONZE_INPATIENT_RAW}")
print(f"Rows written : {inpatient_rows:,}")

print("\nSample rows:")
display(df_inpatient_bronze.limit(5))

In [0]:
# ============================================================
# STEP 3 — INGEST CMS OUTPATIENT DATA
# Same pattern — fetch, add metadata, write to Bronze
# ============================================================

print("=" * 55)
print("  INGESTING CMS OUTPATIENT DATA")
print("=" * 55)

# Fetch all records
outpatient_records = fetch_cms_paginated(
    url     = outpatient_url,
    limit   = API_PAGE_LIMIT,
    timeout = API_TIMEOUT_SECS
)

# Convert to Spark DataFrame
df_outpatient_raw = spark.createDataFrame(outpatient_records)

# Add audit metadata
df_outpatient_bronze = df_outpatient_raw \
    .withColumn("_source",           lit("cms_api_outpatient")) \
    .withColumn("_batch_id",         lit(BATCH_ID)) \
    .withColumn("_batch_date",       lit(BATCH_DATE)) \
    .withColumn("_ingested_at",      lit(BATCH_TIMESTAMP).cast("timestamp")) \
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME)) \
    .withColumn("_pipeline_version", lit(PIPELINE_VERSION))

# Write to Bronze
outpatient_rows = write_delta(
    df         = df_outpatient_bronze,
    table_name = TBL_BRONZE_OUTPATIENT_RAW,
    mode       = "append",
    comment    = "Raw CMS Medicare outpatient claims by provider and APC — append only, never modified"
)

print(f"\nBronze table : {TBL_BRONZE_OUTPATIENT_RAW}")
print(f"Rows written : {outpatient_rows:,}")

print("\nSample rows:")
display(df_outpatient_bronze.limit(5))

In [0]:
# ============================================================
# STEP 4 — VERIFICATION
# Confirm both tables exist and have correct row counts
# ============================================================

print("=" * 55)
print("  BRONZE VERIFICATION")
print("=" * 55)

print("\nInpatient table:")
spark.sql(f"""
    SELECT
        COUNT(*)                  AS total_rows,
        COUNT(DISTINCT _batch_id) AS batches_loaded,
        MIN(_ingested_at)         AS first_ingested,
        MAX(_ingested_at)         AS last_ingested
    FROM {TBL_BRONZE_INPATIENT_RAW}
""").show(truncate=False)

print("Outpatient table:")
spark.sql(f"""
    SELECT
        COUNT(*)                  AS total_rows,
        COUNT(DISTINCT _batch_id) AS batches_loaded,
        MIN(_ingested_at)         AS first_ingested,
        MAX(_ingested_at)         AS last_ingested
    FROM {TBL_BRONZE_OUTPATIENT_RAW}
""").show(truncate=False)

print("Delta history — inpatient:")
spark.sql(f"""
    DESCRIBE HISTORY {TBL_BRONZE_INPATIENT_RAW} LIMIT 3
""").select("version", "timestamp", "operation", "operationMetrics") \
    .show(truncate=False)